# Lunar Incidence Analysis (Legacy)

**Source script:** `lunar_incidence_analysis.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "lunar_incidence"


SEX_ALIASES = ["sex", "gender", "geschlecht"]
NEUTER_ALIASES = ["neuter", "neutered", "castrat", "kastr", "spayed", "steril", "reproductive"]

MOON_FEATURE_TARGETS = [
    "num__moon_phase_sin",
    "num__moon_phase_cos",
    "num__moon_phase_fraction_illuminated",
]


@dataclass
class ColumnMap:
    outcome: Optional[str]
    date: Optional[str]
    sex: Optional[str]
    neuter: Optional[str]
    moon_angle: Optional[str]
    moon_frac: Optional[str]
    moon_cat: Optional[str]
    is_full: Optional[str]
    is_new: Optional[str]
    moon_sin: Optional[str]
    moon_cos: Optional[str]


def setup_logging() -> logging.Logger:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    return logging.getLogger("lunar_incidence")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Moon-focused incidence/normalization analysis.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Input CSV/XLSX file.")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed.")
    parser.add_argument("--bootstrap", type=int, default=2000, help="Bootstrap repeats.")
    parser.add_argument("--window-days", type=int, default=2, help="Window +/- days for near full/new analyses.")
    parser.add_argument(
        "--adjust-seasonality",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Adjust logistic screening models for seasonality (sin/cos day-of-year).",
    )
    parser.add_argument(
        "--adjust-sex-neuter",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Adjust logistic screening models for sex/neuter (unknown retained).",
    )
    return parser.parse_args()


def normalize_key(text: str) -> str:
    return "".join(ch for ch in str(text).lower() if ch.isalnum())


def find_column(df: pd.DataFrame, exact_candidates: Sequence[str], contains_candidates: Sequence[str]) -> Optional[str]:
    cols = list(df.columns)
    norm_to_col = {normalize_key(c): c for c in cols}

    for c in exact_candidates:
        if c in df.columns:
            return c
        c_norm = normalize_key(c)
        if c_norm in norm_to_col:
            return norm_to_col[c_norm]

    for c in cols:
        c_norm = normalize_key(c)
        for token in contains_candidates:
            if normalize_key(token) in c_norm:
                return c
    return None


def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported input format: {path.suffix}")


def detect_columns(df: pd.DataFrame) -> ColumnMap:
    return ColumnMap(
        outcome=find_column(df, ["group", "diagnosis_group", "outcome_group", "outcome"], ["group"]),
        date=find_column(df, ["presentation_date", "date of presentation", "presentation date"], ["presentationdate", "date"]),
        sex=find_column(df, SEX_ALIASES, SEX_ALIASES),
        neuter=find_column(df, ["neuter status"], NEUTER_ALIASES),
        moon_angle=find_column(df, ["moon_phase_angle_deg"], ["moonphaseangle", "moon_phase_angle"]),
        moon_frac=find_column(df, ["moon_phase_fraction_illuminated"], ["fractionilluminated", "moon_phase_fraction"]),
        moon_cat=find_column(df, ["moon_phase_category"], ["moonphasecategory", "moon_phase_category"]),
        is_full=find_column(df, ["is_full_moon"], ["isfullmoon", "full_moon"]),
        is_new=find_column(df, ["is_new_moon"], ["isnewmoon", "new_moon"]),
        moon_sin=find_column(df, ["moon_phase_sin"], ["moonphasesin"]),
        moon_cos=find_column(df, ["moon_phase_cos"], ["moonphasecos"]),
    )


def map_moon_category_value(v: object) -> str:
    s = str(v).strip().lower()
    if s in {"nan", "none", "", "missing", "unknown", "unk"}:
        return "unknown"
    if "new" in s:
        return "new"
    if "full" in s:
        return "full"
    if "wax" in s:
        return "waxing"
    if "wan" in s:
        return "waning"
    return "unknown"


def map_sex(v: object) -> str:
    s = str(v).strip().lower()
    if s in {"female", "f", "weiblich", "w"}:
        return "female"
    if s in {"male", "m", "männlich", "maennlich"}:
        return "male"
    if s in {"", "nan", "none", "unknown", "unbekannt", "?", "missing"}:
        return "unknown"
    return "unknown"


def map_neuter(v: object) -> str:
    s = str(v).strip().lower()
    neutered_tokens = {"neutered", "spayed", "castrated", "kastriert", "sterilisiert"}
    intact_tokens = {"intact", "entire", "unkastriert", "nicht kastriert"}
    if s in neutered_tokens:
        return "neutered"
    if s in intact_tokens:
        return "intact"
    if s in {"", "nan", "none", "unknown", "unbekannt", "missing"}:
        return "unknown"
    return "unknown"


def normalize_categories(df: pd.DataFrame, cols: ColumnMap, logger: logging.Logger) -> Tuple[pd.DataFrame, Dict[str, bool]]:
    out = df.copy()

    flags = {
        "has_outcome": cols.outcome is not None,
        "has_date": cols.date is not None,
        "seasonality_enabled": False,
        "sex_neuter_enabled": False,
    }


    if cols.outcome is None:
        logger.warning("Outcome column could not be detected. Downstream modules will be skipped.")
    if cols.date is not None:
        out["_presentation_date"] = pd.to_datetime(out[cols.date], errors="coerce").dt.normalize()
        flags["has_date"] = out["_presentation_date"].notna().any()
    else:
        out["_presentation_date"] = pd.NaT
        flags["has_date"] = False
        logger.warning("Date column could not be detected.")


    out["_moon_angle_deg"] = pd.to_numeric(out[cols.moon_angle], errors="coerce") if cols.moon_angle else np.nan
    out["_moon_fraction"] = pd.to_numeric(out[cols.moon_frac], errors="coerce") if cols.moon_frac else np.nan

    if cols.moon_sin and cols.moon_cos:
        out["_moon_sin"] = pd.to_numeric(out[cols.moon_sin], errors="coerce")
        out["_moon_cos"] = pd.to_numeric(out[cols.moon_cos], errors="coerce")
    else:
        theta = np.deg2rad(out["_moon_angle_deg"])
        out["_moon_sin"] = np.sin(theta)
        out["_moon_cos"] = np.cos(theta)
        logger.info("Computed moon_phase_sin/cos from moon_phase_angle_deg.")


    if cols.moon_cat:
        out["_moon_category"] = out[cols.moon_cat].map(map_moon_category_value)
    else:
        out["_moon_category"] = "unknown"


    unknown_mask = out["_moon_category"].eq("unknown")
    if unknown_mask.any():
        illum = out.loc[unknown_mask, "_moon_fraction"]
        ang = out.loc[unknown_mask, "_moon_angle_deg"] % 360.0
        derived = np.where(
            illum >= 0.95,
            "full",
            np.where(
                illum <= 0.05,
                "new",
                np.where(ang < 180.0, "waxing", "waning"),
            ),
        )
        out.loc[unknown_mask, "_moon_category"] = pd.Series(derived, index=out.index[unknown_mask])

    out["_moon_category"] = pd.Categorical(
        out["_moon_category"],
        categories=["new", "waxing", "full", "waning", "unknown"],
        ordered=True,
    )


    if cols.is_full:
        out["_is_full_moon"] = pd.to_numeric(out[cols.is_full], errors="coerce").fillna(0).astype(int).clip(0, 1)
    else:
        out["_is_full_moon"] = (
            out["_moon_category"].astype(str).eq("full") | (pd.to_numeric(out["_moon_fraction"], errors="coerce") >= 0.95)
        ).astype(int)
    if cols.is_new:
        out["_is_new_moon"] = pd.to_numeric(out[cols.is_new], errors="coerce").fillna(0).astype(int).clip(0, 1)
    else:
        out["_is_new_moon"] = (
            out["_moon_category"].astype(str).eq("new") | (pd.to_numeric(out["_moon_fraction"], errors="coerce") <= 0.05)
        ).astype(int)


    if cols.sex is not None:
        out["_sex_norm"] = out[cols.sex].map(map_sex)
    else:
        out["_sex_norm"] = "unknown"
    if cols.neuter is not None:
        out["_neuter_norm"] = out[cols.neuter].map(map_neuter)
    else:
        out["_neuter_norm"] = "unknown"
    out["_sex_neuter"] = out["_sex_norm"].astype(str) + "_" + out["_neuter_norm"].astype(str)
    flags["sex_neuter_enabled"] = True


    if flags["has_date"]:
        doy = out["_presentation_date"].dt.dayofyear.astype(float)
        out["_season_sin"] = np.sin(2.0 * np.pi * doy / 365.25)
        out["_season_cos"] = np.cos(2.0 * np.pi * doy / 365.25)
        out["_month"] = out["_presentation_date"].dt.month
        flags["seasonality_enabled"] = True
    else:
        out["_season_sin"] = np.nan
        out["_season_cos"] = np.nan
        out["_month"] = np.nan
        flags["seasonality_enabled"] = False

    return out, flags


def bootstrap_rr(
    df: pd.DataFrame,
    case_col: str,
    exposure_col: str,
    levels: Sequence[object],
    bootstrap: int,
    seed: int,
) -> Dict[object, Tuple[float, float]]:
    n = len(df)
    if n == 0:
        return {lvl: (np.nan, np.nan) for lvl in levels}
    rng = np.random.default_rng(seed)
    out: Dict[object, Tuple[float, float]] = {}
    for lvl in levels:
        rr_vals = []
        for _ in range(bootstrap):
            idx = rng.integers(0, n, n)
            s = df.iloc[idx]
            baseline = s[case_col].mean()
            exp_n = (s[exposure_col] == lvl).sum()
            if baseline <= 0 or exp_n == 0:
                continue
            risk = s.loc[s[exposure_col] == lvl, case_col].mean()
            rr_vals.append(risk / baseline)
        if rr_vals:
            lo, hi = np.percentile(rr_vals, [2.5, 97.5])
            out[lvl] = (float(lo), float(hi))
        else:
            out[lvl] = (np.nan, np.nan)
    return out


def risk_table_for_levels(
    df: pd.DataFrame,
    case_col: str,
    exposure_col: str,
    levels: Sequence[object],
    bootstrap: int,
    seed: int,
) -> pd.DataFrame:
    baseline = float(df[case_col].mean()) if len(df) else np.nan
    total_cases = int(df[case_col].sum()) if len(df) else 0
    total_n = int(len(df))
    ci = bootstrap_rr(df, case_col, exposure_col, levels, bootstrap=bootstrap, seed=seed)
    rows = []
    for lvl in levels:
        mask = df[exposure_col] == lvl
        exposure_n = int(mask.sum())
        case_n = int(df.loc[mask, case_col].sum()) if exposure_n > 0 else 0
        risk = (case_n / exposure_n) if exposure_n > 0 else np.nan
        rr = (risk / baseline) if (pd.notna(risk) and pd.notna(baseline) and baseline > 0) else np.nan
        lo, hi = ci[lvl]
        rows.append(
            {
                "level": lvl,
                "exposure_n": exposure_n,
                "case_n": case_n,
                "risk": risk,
                "baseline_risk": baseline,
                "total_cases": total_cases,
                "total_n": total_n,
                "risk_ratio": rr,
                "rr_ci_low": lo,
                "rr_ci_high": hi,
            }
        )
    return pd.DataFrame(rows)


def mark_near_anchors(dates: pd.Series, anchor_dates: pd.Series, window_days: int) -> pd.Series:
    if dates.isna().all() or anchor_dates.dropna().empty:
        return pd.Series(np.zeros(len(dates), dtype=int), index=dates.index)
    d = dates.dropna().drop_duplicates().sort_values()
    a = anchor_dates.dropna().drop_duplicates().sort_values()
    if len(a) == 0:
        return pd.Series(np.zeros(len(dates), dtype=int), index=dates.index)


    d_int = d.to_numpy(dtype="datetime64[ns]").astype("int64") // (24 * 3600 * 10**9)
    a_int = a.to_numpy(dtype="datetime64[ns]").astype("int64") // (24 * 3600 * 10**9)
    near_map: Dict[pd.Timestamp, int] = {}
    for dt, dt_i in zip(d.tolist(), d_int.tolist()):
        min_diff = np.min(np.abs(a_int - dt_i))
        near_map[dt] = int(min_diff <= window_days)
    return dates.map(near_map).fillna(0).astype(int)


def run_risk_tables(
    df: pd.DataFrame,
    outcome_col: str,
    bootstrap: int,
    seed: int,
    window_days: int,
    adjust_sex_neuter: bool,
    logger: logging.Logger,
) -> Dict[str, pd.DataFrame]:
    diseases = sorted(df[outcome_col].astype(str).unique().tolist())

    category_rows = []
    fullnew_rows = []
    window_rows = []
    strat_rows = []

    cat_levels = ["new", "waxing", "full", "waning"]

    for d_idx, disease in enumerate(diseases):
        case_col = "__case"
        tmp = df.copy()
        tmp[case_col] = (tmp[outcome_col].astype(str) == disease).astype(int)


        t_cat = risk_table_for_levels(
            tmp,
            case_col=case_col,
            exposure_col="_moon_category",
            levels=cat_levels,
            bootstrap=bootstrap,
            seed=seed + 1000 * d_idx + 1,
        )
        t_cat.insert(0, "disease", disease)
        category_rows.append(t_cat)


        for flag_col in ["_is_full_moon", "_is_new_moon"]:
            t_flag = risk_table_for_levels(
                tmp,
                case_col=case_col,
                exposure_col=flag_col,
                levels=[0, 1],
                bootstrap=bootstrap,
                seed=seed + 1000 * d_idx + (10 if flag_col == "_is_full_moon" else 20),
            )
            t_flag.insert(0, "disease", disease)
            t_flag.insert(1, "indicator", flag_col.replace("_", "").lower())
            fullnew_rows.append(t_flag)


        near_full = mark_near_anchors(tmp["_presentation_date"], tmp.loc[tmp["_is_full_moon"] == 1, "_presentation_date"], window_days)
        near_new = mark_near_anchors(tmp["_presentation_date"], tmp.loc[tmp["_is_new_moon"] == 1, "_presentation_date"], window_days)
        tmp["_near_full_window"] = near_full
        tmp["_near_new_window"] = near_new
        for flag_col in ["_near_full_window", "_near_new_window"]:
            t_win = risk_table_for_levels(
                tmp,
                case_col=case_col,
                exposure_col=flag_col,
                levels=[0, 1],
                bootstrap=bootstrap,
                seed=seed + 1000 * d_idx + (30 if flag_col == "_near_full_window" else 40),
            )
            t_win.insert(0, "disease", disease)
            t_win.insert(1, "window_indicator", flag_col.replace("_", "").lower())
            t_win.insert(2, "window_days", window_days)
            window_rows.append(t_win)


        if adjust_sex_neuter:
            for stratum, g in tmp.groupby("_sex_neuter", dropna=False):
                if len(g) < 20:
                    continue
                t_str = risk_table_for_levels(
                    g,
                    case_col=case_col,
                    exposure_col="_moon_category",
                    levels=cat_levels,
                    bootstrap=bootstrap,
                    seed=seed + 1000 * d_idx + abs(hash(str(stratum))) % 997,
                )
                t_str.insert(0, "disease", disease)
                t_str.insert(1, "sex_neuter", stratum)
                strat_rows.append(t_str)

    out = {
        "moon_risk_ratios_by_category": pd.concat(category_rows, ignore_index=True) if category_rows else pd.DataFrame(),
        "moon_risk_ratios_full_new": pd.concat(fullnew_rows, ignore_index=True) if fullnew_rows else pd.DataFrame(),
        "moon_window_risk_ratios": pd.concat(window_rows, ignore_index=True) if window_rows else pd.DataFrame(),
        "moon_risk_ratios_by_category_stratified_sex_neuter": pd.concat(strat_rows, ignore_index=True) if strat_rows else pd.DataFrame(),
    }

    logger.info("Risk tables prepared for %d diseases.", len(diseases))
    return out


def build_design_matrix(
    df: pd.DataFrame,
    include_fraction: bool,
    include_sex_neuter: bool,
    include_seasonality: bool,
) -> Tuple[np.ndarray, List[str], Dict[str, Tuple[float, float]]]:
    x_parts = []
    feature_names: List[str] = []
    scale_params: Dict[str, Tuple[float, float]] = {}


    for c in ["_moon_sin", "_moon_cos"]:
        arr = pd.to_numeric(df[c], errors="coerce")
        arr = arr.fillna(arr.median() if arr.notna().any() else 0.0).to_numpy(dtype=float)
        x_parts.append(arr.reshape(-1, 1))
        feature_names.append(c)


    cont_scaled = []
    if include_fraction:
        cont_scaled.append("_moon_fraction")
    if include_seasonality:
        cont_scaled.extend(["_season_sin", "_season_cos"])

    for c in cont_scaled:
        arr = pd.to_numeric(df[c], errors="coerce")
        arr = arr.fillna(arr.median() if arr.notna().any() else 0.0).to_numpy(dtype=float)
        mu = float(np.mean(arr))
        sd = float(np.std(arr))
        if sd <= 0:
            arr_z = np.zeros_like(arr)
            sd = 1.0
        else:
            arr_z = (arr - mu) / sd
        scale_params[c] = (mu, sd)
        x_parts.append(arr_z.reshape(-1, 1))
        feature_names.append(c)


    if include_sex_neuter:
        dmy = pd.get_dummies(df["_sex_neuter"].fillna("unknown").astype(str), prefix="sex_neuter", drop_first=True, dtype=float)
        if dmy.shape[1] > 0:
            x_parts.append(dmy.to_numpy(dtype=float))
            feature_names.extend(dmy.columns.tolist())

    if not x_parts:
        return np.empty((len(df), 0), dtype=float), [], scale_params

    X = np.concatenate(x_parts, axis=1)
    return X, feature_names, scale_params


def fit_logit_with_bootstrap(
    X: np.ndarray,
    y: np.ndarray,
    feature_names: Sequence[str],
    bootstrap: int,
    seed: int,
) -> Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame]:
    n, p = X.shape
    if n == 0 or p == 0 or np.unique(y).size < 2:
        return {}, pd.DataFrame(), pd.DataFrame()

    model = LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        max_iter=5000,
        random_state=seed,
    )
    model.fit(X, y)

    coef = model.coef_[0]
    intercept = float(model.intercept_[0])
    prob = model.predict_proba(X)[:, 1]
    auc = float(roc_auc_score(y, prob)) if np.unique(y).size == 2 else np.nan


    eps = 1e-12
    p_clip = np.clip(prob, eps, 1 - eps)
    ll_model = float(np.sum(y * np.log(p_clip) + (1 - y) * np.log(1 - p_clip)))
    p0 = float(np.mean(y))
    ll_null = float(np.sum(y * np.log(np.clip(p0, eps, 1 - eps)) + (1 - y) * np.log(np.clip(1 - p0, eps, 1 - eps))))
    pseudo_r2 = float(1 - ll_model / ll_null) if ll_null != 0 else np.nan

    rng = np.random.default_rng(seed)
    coef_boot = np.full((bootstrap, p), np.nan, dtype=float)
    int_boot = np.full(bootstrap, np.nan, dtype=float)
    valid = 0
    for b in range(bootstrap):
        idx = rng.integers(0, n, n)
        yb = y[idx]
        if np.unique(yb).size < 2:
            continue
        mb = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=seed,
        )
        try:
            mb.fit(X[idx], yb)
            coef_boot[b] = mb.coef_[0]
            int_boot[b] = float(mb.intercept_[0])
            valid += 1
        except Exception:
            continue

    term_rows = []
    for i, name in enumerate(feature_names):
        c = float(coef[i])
        boot_col = coef_boot[:, i]
        boot_col = boot_col[np.isfinite(boot_col)]
        if len(boot_col) > 1:
            lo, hi = np.percentile(boot_col, [2.5, 97.5])
        else:
            lo, hi = np.nan, np.nan
        term_rows.append(
            {
                "term": name,
                "coef": c,
                "odds_ratio": float(np.exp(c)),
                "coef_ci_low": float(lo) if pd.notna(lo) else np.nan,
                "coef_ci_high": float(hi) if pd.notna(hi) else np.nan,
                "or_ci_low": float(np.exp(lo)) if pd.notna(lo) else np.nan,
                "or_ci_high": float(np.exp(hi)) if pd.notna(hi) else np.nan,
                "bootstrap_valid_draws": valid,
            }
        )


    joint_rows = []
    if "_moon_sin" in feature_names and "_moon_cos" in feature_names:
        i_sin = feature_names.index("_moon_sin")
        i_cos = feature_names.index("_moon_cos")
        sin_draw = coef_boot[:, i_sin]
        cos_draw = coef_boot[:, i_cos]
        keep = np.isfinite(sin_draw) & np.isfinite(cos_draw)
        if keep.any():
            sin_draw = sin_draw[keep]
            cos_draw = cos_draw[keep]
            sin_lo, sin_hi = np.percentile(sin_draw, [2.5, 97.5])
            cos_lo, cos_hi = np.percentile(cos_draw, [2.5, 97.5])
            amp = np.sqrt(sin_draw**2 + cos_draw**2)
            amp_lo, amp_hi = np.percentile(amp, [2.5, 97.5])
            sign_consistency = float(
                np.mean((np.sign(sin_draw) == np.sign(coef[i_sin])) & (np.sign(cos_draw) == np.sign(coef[i_cos])))
            )
            periodicity_supported = bool(((sin_lo > 0) or (sin_hi < 0)) and ((cos_lo > 0) or (cos_hi < 0)))
            joint_rows.append(
                {
                    "joint_test": "moon_sin_cos_bootstrap",
                    "sin_ci_low": float(sin_lo),
                    "sin_ci_high": float(sin_hi),
                    "cos_ci_low": float(cos_lo),
                    "cos_ci_high": float(cos_hi),
                    "amplitude_ci_low": float(amp_lo),
                    "amplitude_ci_high": float(amp_hi),
                    "sign_consistency_rate": sign_consistency,
                    "periodicity_supported": periodicity_supported,
                }
            )

    model_summary = {
        "intercept": intercept,
        "auc_fitted": auc,
        "pseudo_r2_mcfadden": pseudo_r2,
        "n_rows": int(n),
        "n_cases": int(y.sum()),
        "bootstrap_valid_draws": int(valid),
    }
    return model_summary, pd.DataFrame(term_rows), pd.DataFrame(joint_rows)


def run_circular_logit(
    df: pd.DataFrame,
    outcome_col: str,
    bootstrap: int,
    seed: int,
    adjust_seasonality: bool,
    adjust_sex_neuter: bool,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[Tuple[str, str], Dict[str, float]], pd.DataFrame]:
    diseases = sorted(df[outcome_col].astype(str).unique().tolist())
    result_rows = []
    summary_rows = []
    model_artifacts: Dict[Tuple[str, str], Dict[str, float]] = {}
    interaction_rows = []

    model_specs = [
        ("model_1_moon_only_circular", False, False, False),
        ("model_2_plus_sex_neuter", False, adjust_sex_neuter, False),
        ("model_3_plus_seasonality", False, False, adjust_seasonality),
        ("model_4_full", True, adjust_sex_neuter, adjust_seasonality),
    ]

    for d_idx, disease in enumerate(diseases):
        y = (df[outcome_col].astype(str) == disease).astype(int).to_numpy(dtype=int)
        for m_idx, (model_name, include_fraction, use_sex_neuter, use_seasonality) in enumerate(model_specs):
            X, feat_names, _ = build_design_matrix(
                df,
                include_fraction=include_fraction,
                include_sex_neuter=use_sex_neuter,
                include_seasonality=use_seasonality,
            )
            if X.shape[1] == 0 or np.unique(y).size < 2:
                continue

            model_summary, term_df, joint_df = fit_logit_with_bootstrap(
                X=X,
                y=y,
                feature_names=feat_names,
                bootstrap=bootstrap,
                seed=seed + 10_000 * d_idx + 100 * (m_idx + 1),
            )
            if term_df.empty:
                continue

            for _, r in term_df.iterrows():
                result_rows.append(
                    {
                        "disease": disease,
                        "model": model_name,
                        "term": r["term"],
                        "coef": r["coef"],
                        "odds_ratio": r["odds_ratio"],
                        "coef_ci_low": r["coef_ci_low"],
                        "coef_ci_high": r["coef_ci_high"],
                        "or_ci_low": r["or_ci_low"],
                        "or_ci_high": r["or_ci_high"],
                        "bootstrap_valid_draws": r["bootstrap_valid_draws"],
                    }
                )

            row = {
                "disease": disease,
                "model": model_name,
                **model_summary,
            }
            if not joint_df.empty:
                for c in joint_df.columns:
                    row[c] = joint_df.iloc[0][c]
            summary_rows.append(row)


            if model_name == "model_1_moon_only_circular":
                lr = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=5000, random_state=seed)
                lr.fit(X, y)
                coef_map = {name: float(v) for name, v in zip(feat_names, lr.coef_[0])}
                model_artifacts[(disease, model_name)] = {
                    "intercept": float(lr.intercept_[0]),
                    "coef_sin": float(coef_map.get("_moon_sin", 0.0)),
                    "coef_cos": float(coef_map.get("_moon_cos", 0.0)),
                }


        if adjust_sex_neuter and "_sex_neuter" in df.columns:
            for stratum, g in df.groupby("_sex_neuter", dropna=False):
                if len(g) < 20:
                    continue
                ys = (g[outcome_col].astype(str) == disease).astype(int).to_numpy(dtype=int)
                if np.unique(ys).size < 2:
                    continue
                Xs, feats, _ = build_design_matrix(g, include_fraction=False, include_sex_neuter=False, include_seasonality=False)
                if Xs.shape[1] == 0:
                    continue
                lr = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=5000, random_state=seed)
                lr.fit(Xs, ys)
                auc = roc_auc_score(ys, lr.predict_proba(Xs)[:, 1]) if np.unique(ys).size == 2 else np.nan
                interaction_rows.append(
                    {
                        "disease": disease,
                        "sex_neuter": stratum,
                        "n": int(len(g)),
                        "cases": int(ys.sum()),
                        "auc_fitted": float(auc),
                        "coef_moon_sin": float(lr.coef_[0][feats.index("_moon_sin")]) if "_moon_sin" in feats else np.nan,
                        "coef_moon_cos": float(lr.coef_[0][feats.index("_moon_cos")]) if "_moon_cos" in feats else np.nan,
                        "note": "Exploratory stratum-specific moon-only model.",
                    }
                )

    logger.info("Circular logistic screening completed for %d diseases.", len(diseases))
    return (
        pd.DataFrame(result_rows),
        pd.DataFrame(summary_rows),
        model_artifacts,
        pd.DataFrame(interaction_rows),
    )


def save_plot_hist(series: pd.Series, title: str, out_file: Path, bins: int = 24) -> None:
    vals = pd.to_numeric(series, errors="coerce").dropna()
    if vals.empty:
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.hist(vals, bins=bins, color="#4C72B0", alpha=0.8, edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel(series.name if series.name else "value")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def save_plot_bar_from_counts(counts: pd.Series, title: str, xlabel: str, out_file: Path) -> None:
    if counts.empty:
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.bar(counts.index.astype(str), counts.values, color="#6BAED6", edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def make_plots(
    df: pd.DataFrame,
    outcome_col: str,
    risk_tables: Dict[str, pd.DataFrame],
    model_artifacts: Dict[Tuple[str, str], Dict[str, float]],
    plots_dir: Path,
) -> None:
    plots_dir.mkdir(parents=True, exist_ok=True)


    save_plot_hist(df["_moon_angle_deg"], "Moon Phase Angle Distribution", plots_dir / "moon_phase_angle_hist.png", bins=36)
    save_plot_hist(
        df["_moon_fraction"],
        "Moon Fraction Illuminated Distribution",
        plots_dir / "moon_fraction_illuminated_hist.png",
        bins=20,
    )
    save_plot_bar_from_counts(
        df["_moon_category"].value_counts(dropna=False).reindex(["new", "waxing", "full", "waning", "unknown"]).fillna(0),
        "Moon Phase Category Counts",
        "moon category",
        plots_dir / "moon_phase_category_counts.png",
    )
    save_plot_bar_from_counts(
        pd.Series(
            {
                "is_full_moon=1": int(df["_is_full_moon"].sum()),
                "is_new_moon=1": int(df["_is_new_moon"].sum()),
            }
        ),
        "Full/New Moon Flag Counts",
        "indicator",
        plots_dir / "full_new_indicator_counts.png",
    )


    cat_df = risk_tables["moon_risk_ratios_by_category"]
    fn_df = risk_tables["moon_risk_ratios_full_new"]
    diseases = sorted(df[outcome_col].astype(str).unique().tolist())

    for disease in diseases:

        sub = cat_df[cat_df["disease"] == disease].copy()
        if not sub.empty:
            sub = sub.set_index("level").reindex(["new", "waxing", "full", "waning"]).reset_index()
            y = sub["risk_ratio"].to_numpy(dtype=float)
            lo = sub["rr_ci_low"].to_numpy(dtype=float)
            hi = sub["rr_ci_high"].to_numpy(dtype=float)
            yerr = np.vstack([np.maximum(0, y - lo), np.maximum(0, hi - y)])

            fig, ax = plt.subplots(figsize=(7.8, 4.8))
            ax.bar(sub["level"].astype(str), y, color="#74C476", edgecolor="black")
            ax.errorbar(sub["level"].astype(str), y, yerr=yerr, fmt="none", ecolor="black", capsize=4)
            ax.axhline(1.0, linestyle="--", color="gray")
            ax.set_title(f"Risk Ratio by Moon Category: {disease}")
            ax.set_xlabel("moon category")
            ax.set_ylabel("risk ratio (vs baseline)")
            fig.tight_layout()
            fig.savefig(plots_dir / f"rr_moon_category_{disease.replace(' ', '_').replace('/', '_')}.png", dpi=180)
            plt.close(fig)


        sub2 = fn_df[(fn_df["disease"] == disease) & (fn_df["level"] == 1)].copy()
        if not sub2.empty:
            labels = sub2["indicator"].astype(str).tolist()
            y = sub2["risk_ratio"].to_numpy(dtype=float)
            lo = sub2["rr_ci_low"].to_numpy(dtype=float)
            hi = sub2["rr_ci_high"].to_numpy(dtype=float)
            yerr = np.vstack([np.maximum(0, y - lo), np.maximum(0, hi - y)])

            fig, ax = plt.subplots(figsize=(7.2, 4.8))
            ax.bar(labels, y, color="#FD8D3C", edgecolor="black")
            ax.errorbar(labels, y, yerr=yerr, fmt="none", ecolor="black", capsize=4)
            ax.axhline(1.0, linestyle="--", color="gray")
            ax.set_title(f"Risk Ratio for Full/New: {disease}")
            ax.set_xlabel("indicator")
            ax.set_ylabel("risk ratio (vs baseline)")
            fig.tight_layout()
            fig.savefig(plots_dir / f"rr_full_new_{disease.replace(' ', '_').replace('/', '_')}.png", dpi=180)
            plt.close(fig)


        key = (disease, "model_1_moon_only_circular")
        if key in model_artifacts:
            art = model_artifacts[key]
            theta = np.linspace(0, 2 * np.pi, 361)
            lin = art["intercept"] + art["coef_sin"] * np.sin(theta) + art["coef_cos"] * np.cos(theta)
            prob = 1 / (1 + np.exp(-lin))

            fig = plt.figure(figsize=(6.2, 6.2))
            ax = fig.add_subplot(111, projection="polar")
            ax.plot(theta, prob, color="#2C7FB8", linewidth=2)
            ax.set_title(f"Predicted Risk vs Lunar Phase: {disease}", va="bottom")
            fig.tight_layout()
            fig.savefig(plots_dir / f"polar_predicted_risk_{disease.replace(' ', '_').replace('/', '_')}.png", dpi=180)
            plt.close(fig)


def write_summary(
    df: pd.DataFrame,
    outcome_col: str,
    risk_tables: Dict[str, pd.DataFrame],
    model_summary_df: pd.DataFrame,
    flags: Dict[str, bool],
    out_file: Path,
    window_days: int,
) -> None:
    lines = []
    lines.append("# Lunar Incidence Analysis Summary")
    lines.append("")
    lines.append("## Scope")
    lines.append("")
    lines.append("This module performs lunar association screening with exposure normalization. It is exploratory and non-causal.")
    lines.append("")
    lines.append("## Dataset")
    lines.append("")
    lines.append(f"- N presentations: {len(df)}")
    lines.append(f"- Outcome column: `{outcome_col}`")
    lines.append("- Disease counts:")
    for k, v in df[outcome_col].astype(str).value_counts().items():
        lines.append(f"  - {k}: {int(v)}")
    lines.append("")
    lines.append("## Exposure Distributions")
    lines.append("")
    moon_counts = df["_moon_category"].value_counts(dropna=False)
    for lvl in ["new", "waxing", "full", "waning", "unknown"]:
        lines.append(f"- moon_category={lvl}: {int(moon_counts.get(lvl, 0))}")
    lines.append(f"- is_full_moon=1: {int(df['_is_full_moon'].sum())}")
    lines.append(f"- is_new_moon=1: {int(df['_is_new_moon'].sum())}")
    lines.append("")
    lines.append("## Key Risk-Ratio Findings")
    lines.append("")

    cat_df = risk_tables["moon_risk_ratios_by_category"].copy()
    if not cat_df.empty:

        cat_df["rr_distance"] = (cat_df["risk_ratio"] - 1.0).abs()
        cat_top = cat_df.sort_values("rr_distance", ascending=False).head(8)
        for _, r in cat_top.iterrows():
            lines.append(
                f"- {r['disease']} | {r['level']}: RR={r['risk_ratio']:.3f} "
                f"(95% CI {r['rr_ci_low']:.3f} to {r['rr_ci_high']:.3f})"
            )
    lines.append("")
    lines.append("## Circular Logistic Screening")
    lines.append("")
    if model_summary_df.empty:
        lines.append("- Circular model results unavailable (insufficient data/columns).")
    else:
        for disease, g in model_summary_df.groupby("disease", sort=True):
            g = g.sort_values("model")
            for _, r in g.iterrows():
                periodic = r.get("periodicity_supported", np.nan)
                lines.append(
                    f"- {disease} | {r['model']}: fitted AUC={r['auc_fitted']:.3f}, "
                    f"pseudo-R2={r['pseudo_r2_mcfadden']:.3f}, "
                    f"periodicity_supported={periodic}"
                )
    lines.append("")
    lines.append("## Sex/Neuter + Seasonality Notes")
    lines.append("")
    lines.append(
        f"- Sex/neuter adjustment enabled: {flags.get('sex_neuter_enabled', False)} "
        f"(unknown retained; never dropped)."
    )
    lines.append(f"- Seasonality adjustment enabled: {flags.get('seasonality_enabled', False)}.")
    lines.append("")
    lines.append("## Window Analysis Note")
    lines.append("")
    lines.append(
        f"- Near-full/new windows use an approximate dataset-timeline definition: "
        f"rows within ±{window_days} days of observed full/new anchor rows."
    )
    lines.append("")
    lines.append("## Limitations")
    lines.append("")
    lines.append("- Observational association screening only; no causal inference.")
    lines.append("- Moon-window proximity is approximate and depends on available row dates.")
    lines.append("- Multiple testing risk across diseases/models/metrics.")
    lines.append("- Internal-only validation context.")
    lines.append("")

    out_file.write_text("\n".join(lines), encoding="utf-8")


def main() -> None:
    args = parse_args()
    logger = setup_logging()
    np.random.seed(args.seed)

    outdir = args.outdir
    plots_dir = outdir / "plots"
    outdir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    logger.info("Loading input: %s", args.input)
    df_raw = load_data(args.input)
    cols = detect_columns(df_raw)
    logger.info("Detected columns: %s", cols)

    df, flags = normalize_categories(df_raw, cols, logger)
    if cols.outcome is None:
        raise ValueError("Outcome column could not be resolved; expected e.g. 'group'.")
    outcome_col = cols.outcome


    if not args.adjust_sex_neuter:
        flags["sex_neuter_enabled"] = False
    if not args.adjust_seasonality:
        flags["seasonality_enabled"] = False


    risk_tables = run_risk_tables(
        df=df,
        outcome_col=outcome_col,
        bootstrap=args.bootstrap,
        seed=args.seed,
        window_days=args.window_days,
        adjust_sex_neuter=flags["sex_neuter_enabled"],
        logger=logger,
    )


    circular_results_df, circular_summary_df, model_artifacts, interaction_df = run_circular_logit(
        df=df,
        outcome_col=outcome_col,
        bootstrap=args.bootstrap,
        seed=args.seed,
        adjust_seasonality=flags["seasonality_enabled"],
        adjust_sex_neuter=flags["sex_neuter_enabled"],
        logger=logger,
    )


    out_map = {
        "moon_risk_ratios_by_category.csv": risk_tables["moon_risk_ratios_by_category"],
        "moon_risk_ratios_full_new.csv": risk_tables["moon_risk_ratios_full_new"],
        "moon_window_risk_ratios.csv": risk_tables["moon_window_risk_ratios"],
        "moon_circular_logit_results.csv": circular_results_df,
        "moon_circular_model_summary.csv": circular_summary_df,
        "moon_risk_ratios_by_category_stratified_sex_neuter.csv": risk_tables[
            "moon_risk_ratios_by_category_stratified_sex_neuter"
        ],
        "moon_interaction_screening_sex_neuter.csv": interaction_df,
    }
    for fname, frame in out_map.items():
        frame.to_csv(outdir / fname, index=False)


    make_plots(
        df=df,
        outcome_col=outcome_col,
        risk_tables=risk_tables,
        model_artifacts=model_artifacts,
        plots_dir=plots_dir,
    )


    summary_md = outdir / "lunar_incidence_summary.md"
    write_summary(
        df=df,
        outcome_col=outcome_col,
        risk_tables=risk_tables,
        model_summary_df=circular_summary_df,
        flags=flags,
        out_file=summary_md,
        window_days=args.window_days,
    )


    logger.info("Outputs written to: %s", outdir)
    top_rr = risk_tables["moon_risk_ratios_by_category"].copy()
    if not top_rr.empty:
        top_rr["abs_rr_dist"] = (top_rr["risk_ratio"] - 1.0).abs()
        top = top_rr.sort_values("abs_rr_dist", ascending=False).head(3)
        logger.info("Top RR deviations (category):")
        for _, r in top.iterrows():
            logger.info(
                "  %s | %s | RR=%.3f (CI %.3f to %.3f)",
                r["disease"],
                r["level"],
                r["risk_ratio"],
                r["rr_ci_low"],
                r["rr_ci_high"],
            )

    logger.info("Done. Summary: %s", summary_md)


if __name__ == "__main__":
    main()
